# Analyse artistes similaires (Feb 2026)

## Charger les deux fichiers CSV sources (artistes similaires et debug).


In [37]:
import pandas as pd
import ast

path_sim = r"../../data/Artistes_Similaires\2026_02_artistes_similaires.csv"
path_dbg = r"../../data/Artistes_Similaires\2026_02_debug.csv"

df_sim = pd.read_csv(path_sim)
df_dbg = pd.read_csv(path_dbg)

# Optionnel: sauvegarde rapide des sources
# df_sim.to_csv(r"../../data/Artistes_Similaires\_df_sim_raw.csv", index=False)
# df_dbg.to_csv(r"../../data/Artistes_Similaires\_df_dbg_raw.csv", index=False)

df_sim.head(), df_dbg.head()


(       Source_Artist        Source_Artist_ID  \
 0            Worakls  5RPzPJCg4ER1LzQkorZ31p   
 1                NTO  7ry8L53T4oJtSIogGYuioq   
 2     Joachim Pastor  6eNOjuJSfKkAvbiGW90AkZ   
 3      Florence Bird  4TNk3AroApHY9cXRR4WD7I   
 4  Sébastien Tellier  23ymPLjbtAMzTJS2qRtQ8Z   
 
                                     Related_Data_Raw  
 0  [{'name': 'Joachim Pastor', 'id': '6eNOjuJSfKk...  
 1  [{'name': 'Joachim Pastor', 'id': '6eNOjuJSfKk...  
 2  [{'name': 'Worakls', 'id': '5RPzPJCg4ER1LzQkor...  
 3  [{'name': 'Philipp Wolf', 'id': '6uKv2ihEYpsDw...  
 4  [{'name': 'Charlotte Gainsbourg', 'id': '2rBcv...  ,
        Input_Name   Selected_Name  Rank  Score  \
 0             ALB             ALB    27    1.0   
 1            Sage            Sage     5    1.0   
 2        Proleter        Proleter     1    1.0   
 3  Caravan Palace  Caravan Palace     1    1.0   
 4       Dimie Cat       Dimie Cat     1    1.0   
 
                               URL            Timestamp  
 

## Normaliser l'encodage des noms pour corriger les caracteres mal decodees (ex: RiviA?re -> Riviere).


In [38]:
def fix_mojibake(s: str) -> str:
    if not isinstance(s, str):
        return s
    # Corrige les s?quences UTF-8 mal d?cod?es en Latin-1
    if "Ã" in s or "Â" in s:
        try:
            return s.encode("latin1").decode("utf-8")
        except Exception:
            return s
    return s

# Normalisation des noms (Source_Artist / Input_Name / Selected_Name)
for col in ["Source_Artist"]:
    if col in df_sim.columns:
        df_sim[col] = df_sim[col].map(fix_mojibake)

for col in ["Input_Name", "Selected_Name"]:
    if col in df_dbg.columns:
        df_dbg[col] = df_dbg[col].map(fix_mojibake)

# Optionnel: sauvegarde apres normalisation
# df_sim.to_csv(r"../../data/Artistes_Similaires\_df_sim_norm.csv", index=False)
# df_dbg.to_csv(r"../../data/Artistes_Similaires\_df_dbg_norm.csv", index=False)

(df_sim["Source_Artist"].head(), df_dbg[["Input_Name", "Selected_Name"]].head())


(0              Worakls
 1                  NTO
 2       Joachim Pastor
 3        Florence Bird
 4    Sébastien Tellier
 Name: Source_Artist, dtype: object,
        Input_Name   Selected_Name
 0             ALB             ALB
 1            Sage            Sage
 2        Proleter        Proleter
 3  Caravan Palace  Caravan Palace
 4       Dimie Cat       Dimie Cat)

Recuperer les Input_Name dont le Score est different de 1.


In [39]:
# Recuperer Input_Name avec Score != 1

df_dbg["Score"] = pd.to_numeric(df_dbg["Score"], errors="coerce")
input_name_score_not_1 = df_dbg.loc[df_dbg["Score"] != 1, "Input_Name"].dropna().unique()

# Optionnel: sauvegarde
# pd.Series(input_name_score_not_1, name="Input_Name").to_csv(r"../../data/Artistes_Similaires\_input_name_score_not_1.csv", index=False)

print("Nombre d'artistes avec Score != 1 :", len(input_name_score_not_1))
input_name_score_not_1


Nombre d'artistes avec Score != 1 : 45


array(['Sardè Philippe', 'Zambla', 'Monogrenade', 'Ophelie Luminati',
       'Bots Leonard', 'Quentin Bardi', 'Pierre Jean Meric',
       'Nicolas Laroza', 'Elli', 'Kimi Duplizzle', 'Oete',
       'Bird on the Wire', 'Tor', 'Scarecrow Blues Hip Hop',
       'Théodore Paul et Gabriel', 'Jess', 'Ada',
       'Orlando  Cachaíto  López', 'Paris Jazz Big Bang',
       'Fabrice Colombani', 'Christiane Prince', 'Avishai Cohen Trio',
       'Matthias Pothier', 'Poly', 'Multi-interprètes', 'Nu', 'jackie',
       '･－･ ･－ ･･･ ･ －･･', 'SIND', 'Женя Любич', 'RAR',
       "Les Choeurs de l'Armée Rouge", 'Trolle Siebenhaar', 'Lor',
       'Adrie', 'Joshu', 'Dimension Latina', 'Chris Allen',
       'Ballaké Sissoko & Vincent Segal', 'Shelby Lynn Merry',
       'Abdullah Yazbahar', 'Grover Washington', 'CYR JONES',
       'Louis Hardin', 'B O D I E S'], dtype=object)

## Recuperer les Source_Artist dont Source_Artist_ID est vide.


In [40]:
# Recuperer Source_Artist avec Source_Artist_ID vide

mask_empty_id = (
    df_sim["Source_Artist_ID"].isna()
    | (df_sim["Source_Artist_ID"].astype(str).str.strip() == "")
    | (df_sim["Source_Artist_ID"].astype(str).str.strip() == "[]")
)

source_artist_empty_id = df_sim.loc[mask_empty_id, "Source_Artist"].dropna().unique()

# Optionnel: sauvegarde
# pd.Series(source_artist_empty_id, name="Source_Artist").to_csv(r"../../data/Artistes_Similaires\_source_artist_empty_id.csv", index=False)

print("Nombre d'artistes avec Source_Artist_ID vide :", len(source_artist_empty_id))
source_artist_empty_id


Nombre d'artistes avec Source_Artist_ID vide : 114


array(['Adem And One Little Plane',
       'Laurent Garnier feat. The L.B.S. Crew', 'The L.B.S Crew',
       'Dj Rocca Present Erodiscotique', 'Gordon Tracks',
       'Jeanette Robertson', 'Sardè Philippe', 'André Pascal',
       'Adele Haenel', 'Jean-Pierre Castaldi', 'Dinno', 'Monogrenade',
       'Niki Demiller', 'Jean Claude Brialy', 'Ophelie Luminati',
       'Bots Leonard', 'Pierre Jean Meric', 'Nicolas Laroza',
       'Amina Cadelli', 'Camille Faure', 'Ottis Cœur', 'Kimi Duplizzle',
       'Abou', 'Oete', 'Bristol', 'Scarecrow Blues Hip Hop',
       'Carlos González', 'Paris Jazz Big Bang', 'Jérôme Perez',
       'Fabrice Colombani', 'Lipka', 'The Rosenberg Trio', 'The Van Jets',
       'Lindstrøm', 'Debbie Stoockett alias Nicole Croisille',
       'Abdelouahab Sefsaf', 'Giulia & Los Tellarini', 'Reyvegui',
       'Phoebe Killdeer & The Short Straws', 'Camille Bernon',
       'Dorian and The Dawn Riders', 'Vaska Jankovska',
       'One Fine Day Kids', 'Multi-interprètes', 'Adam

## Extraire les N premiers Related_Data_Raw, compter les artistes (value_counts) et produire un DataFrame.


In [ ]:
# Extraire les name des N premiers Related_Data_Raw

N = 7  # parametrable
USE_ALL_ROWS_IF_EMPTY = True  # fallback si aucun Related_Data_Raw pour les IDs vides

def parse_related(raw):
    if pd.isna(raw) or str(raw).strip() == "":
        return []
    try:
        data = ast.literal_eval(raw)
        if isinstance(data, list):
            return data
    except Exception:
        return []
    return []

# Sous-ensemble ou l'ID est vide
_df_empty = df_sim.loc[mask_empty_id].copy()

if _df_empty["Related_Data_Raw"].notna().sum() == 0:
    print("Aucun Related_Data_Raw pour les Source_Artist_ID vides.")
    if USE_ALL_ROWS_IF_EMPTY:
        print("Fallback: utilisation de toutes les lignes pour Related_Data_Raw.")
        _df_related = df_sim.copy()
    else:
        _df_related = _df_empty
else:
    _df_related = _df_empty

names_list = []
for raw in _df_related["Related_Data_Raw"].dropna():
    related = parse_related(raw)
    top_n = related[:N]
    for d in top_n:
        if isinstance(d, dict) and "name" in d:
            names_list.append(fix_mojibake(d.get("name")))

vc_names = pd.Series(names_list).dropna().value_counts()

# DataFrame propre (pas d'artistes en index)
vc_names_df = vc_names.rename("Count").reset_index().rename(columns={"index": "Artist"})

# Optionnel: sauvegarde
vc_names_df.Artist.to_csv(r"../../data/Artistes_Similaires\_related_topN_counts.csv", index=False)

vc_names_df.head(20)


Aucun Related_Data_Raw pour les Source_Artist_ID vides.
Fallback: utilisation de toutes les lignes pour Related_Data_Raw.


,Artist,Count
0,Melissa Laveaux,28
1,Mari Samuelsen,21
2,Louisa Fuller,20
3,Malik Djoudi,20
4,Anthony Weeden,20
5,H-Burns,19
6,Villagers,18
7,Tim Baker,18
8,Kevin Morby,18
9,Sergiu Schwartz,17


## Filtrer avec le seuil M, exclure les Source_Artist deja presents, puis exporter le resultat en CSV.


In [ ]:
# Filtrer les artistes avec seuil M et exclure Source_Artist

M = 4  # parametrable

if vc_names_df.empty:
    print("vc_names_df est vide. Verifie le Related_Data_Raw et/ou le fallback.")
    filtered_artists_df = vc_names_df
else:
    source_artist_set = set(df_sim["Source_Artist"].dropna().unique())

    filtered_artists_df = vc_names_df[vc_names_df["Count"] >= M]
    filtered_artists_df = filtered_artists_df[~filtered_artists_df["Artist"].isin(source_artist_set)]

    # Export CSV (sans index)
    filtered_artists_df.Artist.to_csv(r"../../data/Artistes_Similaires\filtered_artists.csv", index=False)

filtered_artists_df


,Artist,Count
17,Lockhart,16
20,Alan Raph,16
32,Lancelot,15
34,MGMT,15
40,Jeanne Balibar,15
...,...,...
2013,Stereoclip,4
2015,Moses Gunn Collective,4
2016,Kali Uchis,4
2018,Michel Petrossian,4
